# V08 - Transformations: do dado de origem ao dado curado

**Objetivo:** inspecionar Transformations existentes e demonstrar o ciclo de uma Transformation de teste com destino DMS (criar -> recuperar -> excluir). A **execucao** (`run`) e uma etapa explicita e separada, desabilitada por padrao.

**Por que separar:** criar a definicao e barato e reversivel; executar move dados e exige credenciais e revisao.

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v08',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Inspecionar Transformations existentes
Leitura do catalogo atual, sem executar nada.

In [ ]:
transformations = client.transformations.list(limit=5)
print('transformations visiveis (amostra):', len(transformations))
transformations.to_pandas()[['external_id', 'name', 'conflict_mode']] if len(transformations) else 'Nenhuma transformation visivel.'

## 2. Ciclo de uma Transformation de teste (destino DMS)
Criacao protegida: se faltar permissao de escrita, a celula informa e segue.

In [ ]:
from uuid import uuid4
from cognite.client.data_classes import TransformationWrite, TransformationDestination
from cognite.client.exceptions import CogniteAPIError

run = uuid4().hex[:8]
tr_xid = f'tr_ur_training_v08_{run}'
created_tr = None
draft = TransformationWrite(
    external_id=tr_xid,
    name='UR training - V08 (instances)',
    destination=TransformationDestination.instances(),
    conflict_mode='upsert',
    ignore_null_fields=True,
    query="select 'demo' as externalId  -- consulta minima de demonstracao",
)
print('payload da Transformation (sem executar):')
print(draft.dump())
try:
    created_tr = client.transformations.create(draft)
    print('criada:', created_tr.external_id)
except CogniteAPIError as exc:
    print('Sem permissao para criar Transformation neste ambiente; seguindo apenas com a leitura.')
    print('detalhe:', exc.code)

## 3. Recuperar e (opcional) excluir
A execucao real ficaria aqui, como passo explicito: `client.transformations.run(...)`.

In [ ]:
if created_tr is not None:
    fetched = client.transformations.retrieve(external_id=tr_xid)
    print('recuperada:', fetched.external_id, '| destino:', fetched.destination.type)
    # client.transformations.run(transformation_external_id=tr_xid)  # passo explicito, desabilitado por padrao
    client.transformations.delete(external_id=tr_xid)
    print('excluida:', tr_xid)
else:
    print('Nada a recuperar/excluir: a Transformation de teste nao foi criada.')

## Mini-exercicio
- Troque o destino para `TransformationDestination.nodes(view=...)` apontando para uma view real e reescreva a `query` para projetar `externalId` e `name`.
- Descreva (markdown) qual `conflict_mode` (`upsert`, `update`, `abort`, `delete`) voce usaria em uma carga incremental e por que.